# Stereo Visual Odometry on KITTI — visual walkthrough

Each stage of the pipeline, one figure at a time. The logic lives in the
`stereo_slam` package; this notebook only calls into it.

Set `KITTI_ROOT` (or edit `DATA_ROOT` below) before running.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)

DATA_ROOT = os.environ.get("KITTI_ROOT")  # or hardcode a path here
SEQUENCE = "00"

## 1. Load a sequence

Images, calibration and ground-truth poses.

In [ ]:
from stereo_slam import KittiSequence

sequence = KittiSequence(SEQUENCE, data_root=DATA_ROOT, low_memory=True)
print(sequence)
print("P0 (left projection):\n", sequence.P0.round(2))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6))
axes[0].imshow(sequence.left(0), cmap="gray");  axes[0].set_title("left camera, frame 0")
axes[1].imshow(sequence.right(0), cmap="gray"); axes[1].set_title("right camera, frame 0")
for ax in axes: ax.axis("off")

## 2. Disparity

`StereoBM` is a local block matcher: fast, noisy on the low-texture road.
`StereoSGBM` adds a semi-global smoothness cost and is visibly cleaner at a
real time cost. The black band on the left is the region with no valid
disparity — the first `numDisparities` columns have no counterpart inside the
search range.

In [ ]:
from stereo_slam.stereo import compute_disparity

left, right = sequence.left(0), sequence.right(0)
fig, axes = plt.subplots(2, 1, figsize=(14, 7))
for ax, matcher in zip(axes, ["bm", "sgbm"]):
    disparity = compute_disparity(left, right, matcher=matcher)
    im = ax.imshow(disparity); ax.set_title(f"disparity — Stereo{matcher.upper()}"); ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.02)

## 3. Depth

`Z = f·b/d`. Where the stereo match fails the disparity collapses toward zero
and the depth explodes — the histogram shows the real scene depths bunched at
the low end and a spike of nonsense at the top. Everything above `max_depth`
is dropped before pose estimation.

In [ ]:
from stereo_slam.stereo import stereo_to_depth, left_border_mask

depth = stereo_to_depth(left, right, sequence.P0, sequence.P1, matcher="sgbm")

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
im = axes[0].imshow(np.clip(depth, 0, 100)); axes[0].set_title("depth (clipped at 100 m)"); axes[0].axis("off")
fig.colorbar(im, ax=axes[0], fraction=0.03)
axes[1].hist(depth.flatten(), bins=100); axes[1].set_yscale("log")
axes[1].set_title("depth distribution"); axes[1].set_xlabel("depth [m]")

In [ ]:
mask = left_border_mask(sequence.imheight, sequence.imwidth)
plt.imshow(mask, cmap="gray"); plt.title("feature detection mask"); plt.axis("off");

## 4. Features and matching

ORB is binary and fast (Hamming distance); SIFT is float, slower, and more
repeatable. Lowe's ratio test keeps only matches whose nearest neighbour
clearly beats the second — the single most effective outlier filter here.

In [ ]:
from stereo_slam.features import extract_features, match_features, filter_matches, visualize_matches

next_left = sequence.left(1)
for detector in ["orb", "sift"]:
    kp0, des0 = extract_features(left, detector, mask)
    kp1, des1 = extract_features(next_left, detector, mask)
    raw = match_features(des0, des1, detector=detector)
    good = filter_matches(raw, ratio=0.5)
    print(f"{detector}: {len(kp0)} keypoints, {len(raw)} matches -> {len(good)} after ratio test")
    visualize_matches(left, kp0, next_left, kp1, good[:60])
    plt.title(f"{detector.upper()} matches, frames 0 -> 1")

## 5. Motion estimation

Matched keypoints in frame 0 are back-projected to 3D with the depth map, then
solved against their 2D positions in frame 1 by PnP with RANSAC. This is where
the stereo rig earns its keep: monocular essential-matrix decomposition would
give the same rotation but no absolute scale.

In [ ]:
from stereo_slam.dataset import decompose_projection_matrix
from stereo_slam.odometry import estimate_motion, to_homogeneous

k_left, _, _ = decompose_projection_matrix(sequence.P0)
kp0, des0 = extract_features(left, "orb", mask)
kp1, des1 = extract_features(next_left, "orb", mask)
matches = filter_matches(match_features(des0, des1, detector="orb"), 0.5)

rmat, tvec, _, _ = estimate_motion(matches, kp0, kp1, k_left, depth)
print("estimated frame 0 -> 1 transform:\n", to_homogeneous(rmat, tvec).round(4))
print("\nground-truth pose of frame 1:\n", sequence.gt[1].round(4))

## 6. Full run and evaluation

A few hundred frames is enough to see the drift behaviour; drop `num_frames`
for the whole sequence. Use short `lengths` for the KITTI metric on a short run,
otherwise no sub-sequence fits.

In [ ]:
from stereo_slam import run_slam
from stereo_slam.metrics import evaluate, format_report

N = 300
with_loop = run_slam(sequence, detector="orb", stereo_matcher="bm",
                     loop_closure=True, num_frames=N, verbose=False)
without_loop = run_slam(sequence, detector="orb", stereo_matcher="bm",
                        loop_closure=False, num_frames=N, verbose=False)

In [ ]:
gt = sequence.gt[:N]
for label, result in [("with loop closure", with_loop), ("odometry only", without_loop)]:
    print(format_report(evaluate(gt, result["trajectory"], lengths=(10, 20, 50, 100)), title=label))
    print()

In [ ]:
from stereo_slam.plotting import plot_trajectory_2d, plot_error_over_distance

plot_trajectory_2d({"Ground truth": gt,
                    "VO + loop closure": with_loop["trajectory"],
                    "VO only": without_loop["trajectory"]},
                   title=f"KITTI {SEQUENCE}, first {N} frames",
                   loop_events=with_loop["loop_closer"].loop_events)
plot_error_over_distance(gt, without_loop["trajectory"]);

## What this run shows

Error grows monotonically with distance travelled — the signature of
open-loop odometry with no bundle adjustment. Loop closure removes the offset
at the instant of detection but does nothing to the poses in between, which is
why the improvement in the aggregate metrics is modest. See the README's
Limitations section.